# Week 19: Time Series, State Space Models & Mamba
## Notebook 07: PatchTST: Subseries Patching & Channel Independence

### 1. Conceptual Intuition & Theoretical Background
Applying standard Vision/NLP Transformers directly to 1D time series (treating each timestamp as an individual token) causes two severe problems:
1. A single timestamp (scalar price or temperature) carries minimal semantic information compared to a word or image patch.
2. Computational complexity scales quadratically with sequence length: $O(L^2)$.

**PatchTST** introduces two paradigm shifts:
1. **Subseries Patching:** Consecutive timestamps are grouped into overlapping patches of length $P$ with stride $S$:
$$N = \left\lfloor \frac{L - P}{S} \right\rfloor + 1$$
This extracts local semantic patterns while reducing attention complexity by a factor of $(P / S)^2$.

2. **Channel Independence (CI):** Multivariate time series contain $M$ variables. Rather than cross-attending across variables, all $M$ series are processed independently through a **shared Transformer backbone**:
- Eliminates cross-variable noise and overfitting.
- Outperforms complex cross-variable models across standard benchmarks (ETTh, Weather, Electricity).

---

### 2. Scratch Python Implementation: PatchTST Unfolding & Attention

In [ ]:
import numpy as np

def create_time_series_patches(x, patch_len=16, stride=8):
    """
    x: 1D time series array of shape (seq_len,)
    Returns: Patches of shape (num_patches, patch_len)
    """
    seq_len = len(x)
    num_patches = (seq_len - patch_len) // stride + 1
    patches = []
    for i in range(num_patches):
        start = i * stride
        patches.append(x[start : start + patch_len])
    return np.array(patches)

class ChannelIndependentPatchTSTScratch:
    def __init__(self, patch_len=16, d_model=32, num_heads=2):
        self.patch_len = patch_len
        self.d_model = d_model
        # Linear Projection mapping raw patch -> d_model
        np.random.seed(42)
        self.patch_proj = np.random.normal(0, 0.1, (patch_len, d_model))
        
    def forward_single_channel(self, series, patch_len=16, stride=8):
        # 1. Patching
        patches = create_time_series_patches(series, patch_len, stride)
        num_patches = len(patches)
        
        # 2. Linear projection to d_model
        tokens = np.dot(patches, self.patch_proj) # (N, d_model)
        
        # 3. Self-Attention over patches
        d_k = self.d_model
        scores = np.dot(tokens, tokens.T) / np.sqrt(d_k)
        weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        weights /= np.sum(weights, axis=-1, keepdims=True)
        
        out = np.dot(weights, tokens)
        return out

# Simulate 3 Multivariate Time Series Channels (e.g. Temperature, Pressure, Humidity)
seq_len = 128
patch_len = 16
stride = 8

channel_1 = np.sin(np.linspace(0, 8*np.pi, seq_len))
channel_2 = np.cos(np.linspace(0, 4*np.pi, seq_len))
channel_3 = np.random.randn(seq_len)

multivariate_data = [channel_1, channel_2, channel_3]

model = ChannelIndependentPatchTSTScratch(patch_len=patch_len, d_model=32)

# Process channels independently through identical shared weights
encoded_channels = [model.forward_single_channel(ch, patch_len, stride) for ch in multivariate_data]

num_patches = (seq_len - patch_len) // stride + 1
print("--- PatchTST Channel Independence ---")
print(f"Original Sequence Length per Channel: {seq_len}")
print(f"Tokens per Channel after Patching:   {num_patches} (Reduced from {seq_len})")
print(f"Encoded Channel 1 Representation:    {encoded_channels[0].shape}")
print(f"Memory Complexity Reduction:         {(seq_len / num_patches)**2:.1f}x speedup in self-attention!")

### 3. Step-by-Step Code Walkthrough
1. **1D Unfolding**: Converts raw scalar streams into localized segment tokens.
2. **Linear Patch Projection**: Maps raw segment values into token embeddings.
3. **Channel-Independent Forwarding**: Every variable shares identical transformer weights without cross-channel parameter leakage.

---

### 4. Industry Applications & Trade-offs
- **Long-Horizon Grid Forecasting:** Predicting electricity loads 720 hours ahead with state-of-the-art accuracy.
- **Trade-off:** Cannot directly model explicit cross-channel interactions when variables are strongly co-dependent.

---

### 5. Top Interview Questions & Answers
**Q1: Why does subseries patching significantly improve time series transformer performance?**
> **Answer:** Patching aggregates local semantic trends into individual tokens and shrinks the sequence length by the stride factor $S$, reducing quadratic attention memory by $(L/S)^2$ and improving long-range receptive fields.

**Q2: What is Channel Independence and why does it often beat multivariate attention?**
> **Answer:** Channel independence treats each multivariate series as an independent univariate channel sharing a single model backbone. It prevents overfitting to spurious inter-channel correlations and handles varying channel counts flexibly.